First define the functions:

In [3]:
import numpy as np

K = 10
T = 10_000
L_ODD = np.array([1,-1,*[0.1 for _ in range(K-2)]])
L_EVEN = np.array([-1,1,*[0.1 for _ in range(K-2)]])

# Q1 and Q3:
def ftl():
    '''this fn returns the ftl regret'''
    cumul_losses = np.zeros(K)
    ftl_loss = 0

    for t in range(1, T):
        I_t = np.argmin(cumul_losses)
        if t%2==0:
            ftl_loss += L_EVEN[I_t]
            cumul_losses += L_EVEN
        else:
            ftl_loss += L_ODD[I_t]
            cumul_losses += L_ODD
    regret = ftl_loss - min(cumul_losses)
    return regret

# Q2 and Q4:
def exp_weights(seed):
    '''returns the exponential weights regret'''
    cumul_losses = np.zeros(K)
    eta = np.sqrt(2*np.log(K)/T)
    experts = np.arange(K)
    p_arr = np.array([1/K]*K)
    rng = np.random.default_rng(seed=seed)
    exp_weight_loss = 0

    for t in range(1, T):
        I_t = rng.choice(experts, size=1, p=p_arr)
        if t%2==0:
            exp_weight_loss += L_EVEN[I_t]
            cumul_losses += L_EVEN
        else:
            exp_weight_loss += L_ODD[I_t]
            cumul_losses += L_ODD

        max_j = eta*max(cumul_losses)
        common_denom = max_j - np.log(np.sum(np.exp(-eta*cumul_losses+max_j)))
        p_arr = np.exp(-eta * cumul_losses + common_denom)

    regret = exp_weight_loss - min(cumul_losses)
    return regret

In [5]:
# Q6:
v_ftl = ftl()
seeds = np.arange(20)
avg_v_ew = np.mean([exp_weights(seed) for seed in seeds])
print('V_TFL = ', round(v_ftl,3))
print('average V_EW = ', round(avg_v_ew,3))

V_TFL =  10000.0
average V_EW =  145.2


In [12]:
def sample_losses():
    x_1,x_2 = np.random.binomial(n=1, p=.5, size=2)
    losses = np.array([x_1-.9*(1-x_1),2*x_2-1,*[0.1 for _ in range(K-2)]])
    return losses

def bernoulli_ftl():
    '''this fn returns the ftl regret'''
    cumul_losses = np.zeros(K)
    ftl_loss = 0

    for t in range(1, T):
        I_t = np.argmin(cumul_losses)
        losses = sample_losses()
        ftl_loss += losses[I_t]
        cumul_losses += losses
    regret = ftl_loss - min(cumul_losses)
    return [ftl_loss, regret]

def bernoulli_exp_weights():
    '''returns the exponential weights regret'''
    cumul_losses = np.zeros(K)
    eta = np.sqrt(2*np.log(K)/T)
    experts = np.arange(K)
    p_arr = np.array([1/K]*K)
    rng = np.random.default_rng()
    exp_weight_loss = 0

    for t in range(1, T):
        I_t = rng.choice(experts, size=1, p=p_arr)
        losses = sample_losses()
        exp_weight_loss += losses[I_t].item()
        cumul_losses += losses

        max_j = eta*max(cumul_losses)
        common_denom = max_j - np.log(np.sum(np.exp(-eta*cumul_losses+max_j)))
        p_arr = np.exp(-eta * cumul_losses + common_denom)

    regret = exp_weight_loss - min(cumul_losses).item()
    return [exp_weight_loss, regret]

In [ ]:
# Q7 + Q8:

l_ftl, v_ftl = np.array([bernoulli_ftl() for _ in range(20)]).mean(0).round(3)
l_ew, v_ew = np.array([bernoulli_exp_weights() for _ in range(20)]).mean(0).round(3)
print('average L_FTL = ', l_ftl, ', average V_FTL = ', v_ftl)
print('average L_EW = ', l_ew, ', average V_EW = ', v_ew)

average L_FTL =  7.61 , average V_FTL =  14.81
average L_EW =  51.88 , average V_EW =  104.38
